## Import thư viện


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
TARGET = "Attack_type"
PORT_COLUMNS = ["id.orig_p", "id.resp_p"]


## Đọc dữ liệu


In [ ]:
# Đặt train.csv và validation.csv cùng thư mục với notebook.
TRAIN_PATH = r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\project\preprocessing\data\train.csv"
VALIDATION_PATH = r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\project\preprocessing\data\validation.csv"

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Train columns:", len(train_df.columns))
print("Validation columns:", len(validation_df.columns))

assert TARGET in train_df.columns, f"Không tìm thấy {TARGET} trong train"
assert TARGET in validation_df.columns, f"Không tìm thấy {TARGET} trong validation"
assert set(train_df.columns) == set(validation_df.columns), "Train và validation không cùng tập cột"
missing_ports = [c for c in PORT_COLUMNS if c not in train_df.columns]
assert not missing_ports, f"Thiếu cột cổng: {missing_ports}"


## kiểm tra dữ liệu và giá trị của 2 cột cổng


In [ ]:
display(train_df[PORT_COLUMNS].dtypes.to_frame("dtype"))
display(train_df[PORT_COLUMNS].describe(percentiles=[.25, .5, .75]).T)

for col in PORT_COLUMNS:
    print(f"\\n{col}")
    print("Số giá trị thiếu:", train_df[col].isna().sum())
    print("Số giá trị cổng khác nhau:", train_df[col].nunique(dropna=True))
    print("Số giá trị xuất hiện:", train_df[col].notna().sum())


## Phân tích phụ thuộc giữa cổng và Attack_type (chỉ trên train)


## Phân phối nhãn theo các cổng phổ biến


In [ ]:
for col in PORT_COLUMNS:
    print(f"Top 20 giá trị của {col}:")
    top_ports = train_df[col].value_counts(dropna=False).head(20).index
    subset = train_df[train_df[col].isin(top_ports)]
    port_target_pct = pd.crosstab(
        subset[col],
        subset[TARGET],
        normalize="index"
    ) * 100
    display(port_target_pct.round(2))


## Tần suất các cổng phổ biến


In [ ]:
for col in PORT_COLUMNS:
    counts = train_df[col].value_counts().head(20).sort_values()
    plt.figure(figsize=(10, 6))
    counts.plot(kind="barh")
    plt.title(f"Top 20 giá trị {col} trong tập train")
    plt.xlabel("Số lượng mẫu")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## Mức độ liên hệ bằng Mutual Information


In [ ]:
# Mã hóa nhãn thành số nguyên để tính mutual information.
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(train_df[TARGET].astype(str))

mi_rows = []
for col in PORT_COLUMNS:
    # Điền giá trị thiếu bằng một nhãn riêng rồi mã hóa các giá trị cổng.
    port_values = train_df[col].astype("string").fillna("__MISSING__")
    port_codes, uniques = pd.factorize(port_values, sort=True)

    mi = mutual_info_classif(
        port_codes.reshape(-1, 1),
        y_encoded,
        discrete_features=True,
        random_state=RANDOM_STATE
    )[0]

    mi_rows.append({
        "feature": col,
        "unique_port_values": len(uniques),
        "mutual_information": mi
    })

mi_report = pd.DataFrame(mi_rows).sort_values(
    "mutual_information", ascending=False
)
display(mi_report)


### Nhận xét kết quả


## Loại bỏ id.orig_p và id.resp_p


In [ ]:
# Danh sách cột bị loại được cố định theo yêu cầu của E09.
drop_columns = PORT_COLUMNS.copy()

X_train = train_df.drop(columns=[TARGET]).copy()
y_train = train_df[TARGET].copy()

X_validation = validation_df.drop(columns=[TARGET]).copy()
y_validation = validation_df[TARGET].copy()

X_train_processed = X_train.drop(columns=drop_columns)
X_validation_processed = X_validation.drop(columns=drop_columns)

print("Train trước khi loại:", X_train.shape)
print("Train sau khi loại:", X_train_processed.shape)
print("Validation trước khi loại:", X_validation.shape)
print("Validation sau khi loại:", X_validation_processed.shape)

assert not set(drop_columns).intersection(X_train_processed.columns)
assert not set(drop_columns).intersection(X_validation_processed.columns)
assert X_train_processed.columns.tolist() == X_validation_processed.columns.tolist()


## Kiểm tra các cột còn lại


In [ ]:
print("Các cột đã loại:", drop_columns)
print("Số đặc trưng còn lại:", X_train_processed.shape[1])
print("Còn id.orig_p?", "id.orig_p" in X_train_processed.columns)
print("Còn id.resp_p?", "id.resp_p" in X_train_processed.columns)

feature_summary = pd.DataFrame({
    "feature": X_train_processed.columns,
    "dtype": X_train_processed.dtypes.astype(str).values,
    "missing_train": X_train_processed.isna().sum().values,
    "missing_validation": X_validation_processed.isna().sum().values
})
display(feature_summary)


In [ ]:
from pathlib import Path
import json

# Lưu dữ liệu sau bước xử lý của E09
OUTPUT_DIR = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\exp3\E09_RF\E09_Preprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

processed_train = X_train_processed.copy()
processed_train[TARGET] = y_train.to_numpy()

processed_validation = X_validation_processed.copy()
processed_validation[TARGET] = y_validation.to_numpy()

# Lưu dữ liệu CSV trực tiếp vào OUTPUT_DIR
processed_train.to_csv(
    OUTPUT_DIR / "train.csv.gz",
    index=False,
    compression="gzip"
)

processed_validation.to_csv(
    OUTPUT_DIR / "validation.csv.gz",
    index=False,
    compression="gzip"
)

# Lưu cấu hình tiền xử lý JSON cùng thư mục
with (OUTPUT_DIR / "E09_processing_config.json").open("w", encoding="utf-8") as f:
    json.dump({
        "target": TARGET,
        "removed_columns": drop_columns,
        "feature_columns": X_train_processed.columns.tolist(),
    }, f, ensure_ascii=False, indent=2)

print("Đã lưu:", OUTPUT_DIR / "train.csv.gz")
print("Đã lưu:", OUTPUT_DIR / "validation.csv.gz")
print("Đã lưu:", OUTPUT_DIR / "E09_processing_config.json")
